## Framework Overview  

This framework is designed to streamline the **data preprocessing pipeline** by implementing multiple algorithms for:  

- **Data Cleansing**  
- **Outlier Detection & Handling**  
- **Feature Selection**  
- **Model Selection**  

At each step, we evaluate the effectiveness of different algorithms using a **shallow Decision Tree model** and  a **KNN Model**. This allows us to determine the best preprocessing strategy based on performance.  


## Importing Packages

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split #Weak Model Test
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score #Weak Model Test
from sklearn.tree import DecisionTreeClassifier #Weak Model Test
from sklearn.neighbors import KNeighborsClassifier #Weak Model Test
from sklearn.impute import SimpleImputer, KNNImputer #Handle Missing
from sklearn.linear_model import LinearRegression #Handle Missing
from sklearn.experimental import enable_iterative_imputer #Handle Missing(Imported Because of API Change)
from sklearn.impute import IterativeImputer #Handle Missing
from sklearn.neighbors import LocalOutlierFactor #Tame Outlier
from sklearn.ensemble import IsolationForest #Tame Outlier
from scipy.stats import zscore #Tame Outlier
from sklearn.feature_selection import SelectKBest,f_classif,SelectFromModel # Feature Selection
from sklearn.svm import LinearSVC # Feature Selection
from sklearn.ensemble import ExtraTreesClassifier # Feature Selection

## Handle Date Columns

In [2]:
def identify_date_columns(data):
    """ Detect date columns."""
    date_columns=[]
    date_pattern = r'(\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[-/]\d{1,2}[-/]\d{2,4})'

    for col in data.columns:
        if data[col].dtype.name == 'object':
            if all(data[col].str.contains(date_pattern, regex=True, na=True)):
                date_columns.append(col)
    return date_columns

def process_date_columns(data, reference_column=None):
    """ Processes the date columns and extracts useful time-based features."""

    date_columns=identify_date_columns(data)

    for col in date_columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

    for col in date_columns:
        data[f'{col}_year'] = data[col].dt.year.astype(float)
        data[f'{col}_month'] = data[col].dt.month.astype(float)
        data[f'{col}_day'] = data[col].dt.day.astype(float)
        data[f'{col}_day_of_week'] = data[col].dt.dayofweek.astype(float)
        data[f'{col}_hour'] = data[col].dt.hour.astype(float)
        data[f'{col}_day_of_year'] = data[col].dt.dayofyear.astype(float)
        data[f'{col}_is_weekend'] = (data[col].dt.weekday >= 5).astype(float)


    if reference_column:
        for col in date_columns:
            data[f'{col}_time_diff'] = (data[reference_column] - data[col]).dt.total_seconds() / (60 * 60 * 24)

    return data.drop(columns=date_columns)

## Convert into Numeric Values

In [3]:
def ConvertToNumeric(data):
    data=process_date_columns(data)
    cols = data.columns
    num_cols = data._get_numeric_data().columns
    categorical_columns = list(set(cols) - set(num_cols))

    for column in categorical_columns:
        categories = list(data[column].dropna().astype(str).unique())
        data[column] = data[column].map(lambda x: categories.index(str(x)) if str(x) in categories else x)

    return data

## Loading Data

In [4]:
def load_data(path, y_column):
    if("csv" in path):
        data=pd.read_csv(path)
    elif("xlsx" in path):
        data=pd.read_excel(path)
    else:
        print("Format not Supported")
        return None
    data=ConvertToNumeric(data)
    Y=data[y_column]
    X=data.drop(columns=y_column)
    return X,Y

## Implementing Test Models

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

def train_and_evaluate(X, y, expand=False):
    """Trains multiple weak models (3x KNN and 3x Decision Tree) and evaluates them."""

    if len(y) <= 5 or len(X) <= 5:
        return 0

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    models = [
        KNeighborsClassifier(n_neighbors=3),
        KNeighborsClassifier(n_neighbors=5),
        KNeighborsClassifier(n_neighbors=7),

        DecisionTreeClassifier(max_depth=3, random_state=42),
        DecisionTreeClassifier(max_depth=5, random_state=42),
        DecisionTreeClassifier(max_depth=None, min_samples_split=10, random_state=42),

    ]

    all_scores = []

    for i, model in enumerate(models):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=1)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=1)
        f1 = f1_score(y_test, y_pred, average='weighted')

        all_scores.extend([acc, prec, rec, f1])

        if expand:
            print(f"\nModel {i+1}: {model.__class__.__name__}")
            print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")

    weighted_score = sum(all_scores) / len(all_scores)

    return weighted_score


# Handling Missing Data

These functions help clean our dataset by handling missing values efficiently and selecting the best imputation method.

## Available Algorithms  
- **Dropping Methods:**  
  - Drop columns with null values exceeding a threshold  
  - Drop rows with missing values  

- **Statistical Imputation:**  
  - Impute with **mean** or **median**  
  - Impute with **class-specific mean** or **median**  

- **Forward & Backward Filling:**  
  - **Forward fill (ffill)**  
  - **Backward fill (bfill)**  
  - **Interpolate** missing values  

- **Model-Based Imputation:**  
  - **Model Imputation** (predict missing values using a simple model)  
  - **Iterative Model Imputation** (refines predictions iteratively)  
  - **KNN Imputation** (fills missing values based on k-nearest neighbors)  


In [6]:
def drop_rows(X, Y):
    """Drops rows with any missing values."""

    mask=X.notna().all(axis=1)
    return X[mask], Y[mask]

def impute_mean(X, Y):
    """Fills missing values with column mean while leaving non-missing values intact."""

    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()
    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

def impute_median(X, Y):
    """Fills missing values with column median while leaving non-missing values intact."""

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()
    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

def impute_class_mean(X, Y):
    """Fills missing values with the mean of each class in Y, ensuring no NaNs remain."""

    X_copy = X.copy()
    for col in X.columns:
        grouped_means = X.groupby(Y)[col].transform(lambda x: x.mean())
        X_copy[col] = X[col].fillna(grouped_means)  # Fill NaNs with class mean

        if X_copy[col].isna().sum() > 0:
            X_copy[col].fillna(X[col].mean(), inplace=True)

    return X_copy, Y

def impute_class_median(X, Y):
    """Fills missing values with the median of each class in Y."""

    X_copy = X.copy()
    for col in X.columns:
        X_copy[col] = X.groupby(Y)[col].transform(lambda x: x.fillna(x.median()))

        if X_copy[col].isna().sum() > 0:
            X_copy[col].fillna(X[col].mean(), inplace=True)
    return X_copy,Y

def ffill(X, Y):
    """Fills missing values with the previous row (forward fill) and ensures no NaNs remain."""

    X_filled = X.ffill()
    X_filled = X_filled.bfill()
    return X_filled, Y

def bfill(X, Y):
    """Fills missing values with the next row (backward fill)."""

    X_filled = X.bfill()
    X_filled = X_filled.ffill()
    return X_filled,Y

def interpolate(X, Y):
    """Interpolates missing values linearly and ensures no NaNs remain."""

    X_filled = X.interpolate(method="linear", limit_direction="both")
    return X_filled, Y



def Model_imputation(X, Y):
    """Uses a simple regression model to impute missing values."""

    X_copy = X.copy()

    for col in X.columns:
        missing_mask = X_copy[col].isna()
        if missing_mask.sum() > 0:
            known_data = X_copy.dropna()
            known_X = known_data.drop(columns=[col])
            known_y = known_data[col]
            missing_X = X_copy.loc[missing_mask].drop(columns=[col])

            if missing_X.isna().sum().sum() > 0:
                missing_X = missing_X.fillna(known_X.mean())

            if len(known_X) > 0 and len(missing_X) > 0:
                model = LinearRegression()
                model.fit(known_X, known_y)
                X_copy.loc[missing_mask, col] = model.predict(missing_X)

    return X_copy, Y

def Iterative_model_Imputation(X, Y):
    """Uses iterative imputation (sklearn's IterativeImputer) to fill only null values."""

    imputer = IterativeImputer()
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()

    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

def KNN_Imputation(X, Y, n_neighbors=5):
    """Uses KNN to fill missing values while leaving non-missing values intact."""

    imputer = KNNImputer(n_neighbors=n_neighbors)
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()

    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

algorithm_functions_clean_data = {
    "drop_rows": drop_rows,
    "impute_mean": impute_mean,
    "impute_median": impute_median,
    "impute_class_mean": impute_class_mean,
    "impute_class_median": impute_class_median,
    "ffill": ffill,
    "bfill": bfill,
    "interpolate": interpolate,
    "Model_imputation": Model_imputation,
    "Iterative_model_Imputation": Iterative_model_Imputation,
    "KNN_Imputation": KNN_Imputation,
}


In [7]:
def handling_missing_data(X, Y):
    """Applies different missing data handling algorithms, selects the best one, and returns the transformed dataset."""
    if(X.isna().sum().sum()==0):
        return X,Y,"No Null Data",100,{"No Null Data":100}
    Y = Y.dropna(inplace=False)
    X = X.loc[Y.index]
    x_copy = X.copy()
    y_copy = Y.copy()
    acc_holder = {}

    best_algo = None
    best_accuracy = -1

    for name, func in algorithm_functions_clean_data.items():
        X_transformed,Y_transformed = func(x_copy.copy(), y_copy.copy())
        print(name)
        print()
        accuracy = train_and_evaluate(X_transformed, Y_transformed,expand=True)
        acc_holder[name] = accuracy

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_algo = name
    if best_algo:
        x_copy,y_copy = algorithm_functions_clean_data[best_algo](x_copy.copy(), y_copy.copy())

    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

# Taming Outliers

These functions help clean our dataset from outliers efficiently and selecting the best outlier detection algorithm.

## Available Algorithms  
- **IQR Method(1.5):**  
  - Take 1.5 times the IQR and then subtract this value from Q1 and add this value to Q3


- **LOF:**  
  - For any data object **q**, the **LOF score** is computed as the ratio of the **average local density** of its **k-nearest neighbors** to its **own local density** **[25]**.  

  $$
  LOF(q) = \frac{\sum_{x \in N_k(q)} lrd(x)}{|N_k(q)| \times lrd(q)}
  $$

  where the **local reachability density (lrd)** of **q** is given by:  

  $$
  lrd(q) = \frac{|N_k(q)|}{\sum_{x \in N_k(q)} \max(\text{dist}_k(x, D), \text{dist}(q, x))}
  $$

- **SP:**  
   - Employ a **scoring measure** based on the nearest neighbor (**k = 1**) within random sub-samples (**S ⊂ D**).  

    $$  S_p(q) = \min_{{x \in S}} \text{dist}(q, x) $$

    where **dist(q, x)** represents the distance between **q** and **x**.

- **iForest:**  
  - A **random split** is performed on a randomly selected feature.  
  - The partitioning continues until either:  
    - Each node contains only **one data object**, or  
    - The tree reaches its **height limit**.

  $$
  iForest(q) = \frac{1}{t} \sum_{i=1}^{t} l_i(q)
  $$


- **iNNe:**  
  - This method builds **hyperspheres** using all dimensions of the dataset. The **isolation score** of a data object **q** is defined as:  

  $$
  I(q) =
  \begin{cases}
  \tau (\eta_{cnn}(q)), & \text{if } q \in \bigcup_{c \in S} B(c) \\  
  1 - \tau (cnn(q)), & \text{otherwise}  
  \end{cases}
  $$


In [8]:
def IQR(X, Y):
    """Adjust outliers based on IQR min-max whiskers."""
    Q1 = X.quantile(0.25)
    Q3 = X.quantile(0.75)
    IQR = Q3 - Q1
    min_whisker = Q1 - 1.5 * IQR
    max_whisker = Q3 + 1.5 * IQR
    X_adjusted = X.clip(lower=min_whisker, upper=max_whisker, axis=1)
    return X_adjusted, Y

def LOF(X, Y):
    """Adjust only the detected outliers based on LOF anomaly score."""
    clf = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
    X_scores = clf.fit_predict(X)
    adjustment_factor = np.abs(clf.negative_outlier_factor_) / np.max(np.abs(clf.negative_outlier_factor_))

    X_adjusted = X.copy()
    outlier_mask = X_scores == -1
    X_adjusted[outlier_mask] = X[outlier_mask] * (1 - adjustment_factor[outlier_mask, np.newaxis])
    return X_adjusted, Y

def IsolationForestOutlier(X, Y):
    """Only adjust outliers based on Isolation Forest isolation score."""
    iso = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
    preds = iso.fit_predict(X)  # -1 for outliers, 1 for inliers
    scores = iso.decision_function(X)

    adjustment_factor = (scores - scores.min()) / (scores.max() - scores.min())
    X_adjusted = X.copy()

    for i, pred in enumerate(preds):
        if pred == -1:
            X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])

    return X_adjusted, Y



def SP(X, Y):
    """Adjust only outliers beyond 3 std dev using Standardization Projection."""
    X_adjusted = X.copy()
    Z_scores = np.abs(zscore(X, nan_policy='omit'))
    Z_scores = np.nan_to_num(Z_scores, nan=0)

    for col in X.columns:
        col_idx = X.columns.get_loc(col)
        for i in range(len(X)):
            if Z_scores[i, col_idx] > 3:
                factor = 3 / Z_scores[i, col_idx]
                X_adjusted.iloc[i, col_idx] = X.iloc[i, col_idx] * factor

    return X_adjusted, Y


def IsolationNNe(X, Y):
    """Only adjust outliers based on Isolation Forest distance score."""
    iso = IsolationForest(contamination=0.05, n_estimators=200, random_state=42)
    preds = iso.fit_predict(X)
    dist = iso.decision_function(X)
    adjustment_factor = np.abs(dist) / np.max(np.abs(dist))

    X_adjusted = X.copy()
    for i, pred in enumerate(preds):
        if pred == -1:
            X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])

    return X_adjusted, Y


algorithm_functions_tame_outlier = {
    "IQR": IQR,
    "LOF": LOF,
    "iForest": IsolationForestOutlier,
    "SP": SP,
    "iNNe": IsolationNNe,
}


In [9]:
def taming_outliers(X,Y):
    acc_holder = {}
    x_copy = X.copy()
    y_copy = Y.copy()
    best_algo = None
    best_accuracy = 0


    for name, func in algorithm_functions_tame_outlier.items():
        X_Adjusted,Y_Adjusted = func(x_copy.copy(), y_copy.copy())
        print(name)
        print()
        accuracy = train_and_evaluate(X_Adjusted, Y_Adjusted,expand=True)
        acc_holder[name] = accuracy


        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_algo = name
    if best_algo:
        x_copy,y_copy = algorithm_functions_tame_outlier[best_algo](x_copy.copy(), y_copy.copy())

    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

In [10]:
# X,Y=load_data(path='Shariati_final_results.xlsx',y_column='کووید 19-RT-PCR(COVID-19)')
# X,Y=load_data(path="data1_1.csv",y_column="smokingstutus")
X,Y=load_data(path="./drive/MyDrive/healthcare_dataset.csv",y_column="Test Results")

<ipython-input-2-346fd3ee0976>:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  if all(data[col].str.contains(date_pattern, regex=True, na=True)):


In [11]:
X.columns

,Name,Age,Gender,Blood Type,Medical Condition,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,...,Date of Admission_hour,Date of Admission_day_of_year,Date of Admission_is_weekend,Discharge Date_year,Discharge Date_month,Discharge Date_day,Discharge Date_day_of_week,Discharge Date_hour,Discharge Date_day_of_year,Discharge Date_is_weekend
0,0,30,0,0,0,0,0,0,18856.281306,328,...,0.0,31.0,0.0,2024.0,2.0,2.0,4.0,0.0,33.0,0.0
1,1,62,0,1,1,1,1,1,33643.327287,265,...,0.0,232.0,0.0,2019.0,8.0,26.0,0.0,0.0,238.0,0.0
2,2,76,1,2,1,2,2,2,27955.096079,205,...,0.0,265.0,0.0,2022.0,10.0,7.0,4.0,0.0,280.0,0.0
3,3,28,1,3,2,3,3,1,37909.782410,450,...,0.0,323.0,0.0,2020.0,12.0,18.0,4.0,0.0,353.0,0.0
4,4,43,1,4,0,4,4,2,14238.317814,458,...,0.0,262.0,0.0,2022.0,10.0,9.0,6.0,0.0,282.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,25707,42,1,3,3,22564,21930,0,2650.714952,417,...,0.0,229.0,1.0,2020.0,9.0,15.0,1.0,0.0,259.0,0.0
55496,805,61,1,5,1,801,795,4,31457.797307,316,...,0.0,23.0,0.0,2020.0,2.0,1.0,5.0,0.0,32.0,1.0
55497,1890,38,1,6,4,1865,1844,3,27620.764717,347,...,0.0,195.0,0.0,2020.0,8.0,10.0,0.0,0.0,223.0,0.0
55498,25997,43,0,7,5,22794,22164,1,32451.092358,321,...,0.0,145.0,1.0,2019.0,5.0,31.0,4.0,0.0,151.0,0.0


In [12]:
x_copy_clean_data, y_copy_clean_data, best_algo_clean_data, best_accuracy_clean_data,acc_holder_clean_data=handling_missing_data(X,Y)

In [13]:
x_copy_clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Name                           55500 non-null  int64  
 1   Age                            55500 non-null  int64  
 2   Gender                         55500 non-null  int64  
 3   Blood Type                     55500 non-null  int64  
 4   Medical Condition              55500 non-null  int64  
 5   Doctor                         55500 non-null  int64  
 6   Hospital                       55500 non-null  int64  
 7   Insurance Provider             55500 non-null  int64  
 8   Billing Amount                 55500 non-null  float64
 9   Room Number                    55500 non-null  int64  
 10  Admission Type                 55500 non-null  int64  
 11  Medication                     55500 non-null  int64  
 12  Date of Admission_year         55500 non-null 

In [14]:
x_copy_tame_outlier, y_copy_tame_outlier, best_algo_tame_outlier, best_accuracy_tame_outlier,acc_holder_tame_outlier=taming_outliers(x_copy_clean_data,y_copy_clean_data)


IQR


Model 1: KNeighborsClassifier
  Accuracy: 0.3738, Precision: 0.3758, Recall: 0.3738, F1-score: 0.3688

Model 2: KNeighborsClassifier
  Accuracy: 0.3680, Precision: 0.3705, Recall: 0.3680, F1-score: 0.3626

Model 3: KNeighborsClassifier
  Accuracy: 0.3638, Precision: 0.3646, Recall: 0.3638, F1-score: 0.3624

Model 4: DecisionTreeClassifier
  Accuracy: 0.3348, Precision: 0.3150, Recall: 0.3348, F1-score: 0.2606

Model 5: DecisionTreeClassifier
  Accuracy: 0.3301, Precision: 0.3238, Recall: 0.3301, F1-score: 0.2746

Model 6: DecisionTreeClassifier
  Accuracy: 0.4048, Precision: 0.4049, Recall: 0.4048, F1-score: 0.4047


<ipython-input-8-906203b375ce>:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.92760743e+00 2.87835574e+00 3.84704344e+00 ... 4.58375746e+03
 3.23586097e+04 6.83337609e+03]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted[outlier_mask] = X[outlier_mask] * (1 - adjustment_factor[outlier_mask, np.newaxis])
<ipython-input-8-906203b375ce>:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[73.24908226 26.86465359 41.35571695 ... 14.30338295 76.5906361
 49.1143115 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted[outlier_mask] = X[outlier_mask] * (1 - adjustment_factor[outlier_mask, np.newaxis])
<ipython-input-8-906203b375ce>:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error i

LOF


Model 1: KNeighborsClassifier
  Accuracy: 0.3704, Precision: 0.3723, Recall: 0.3704, F1-score: 0.3651

Model 2: KNeighborsClassifier
  Accuracy: 0.3679, Precision: 0.3705, Recall: 0.3679, F1-score: 0.3627

Model 3: KNeighborsClassifier
  Accuracy: 0.3650, Precision: 0.3657, Recall: 0.3650, F1-score: 0.3635

Model 4: DecisionTreeClassifier
  Accuracy: 0.3382, Precision: 0.3227, Recall: 0.3382, F1-score: 0.2118

Model 5: DecisionTreeClassifier
  Accuracy: 0.3297, Precision: 0.3210, Recall: 0.3297, F1-score: 0.2704

Model 6: DecisionTreeClassifier
  Accuracy: 0.4057, Precision: 0.4059, Recall: 0.4057, F1-score: 0.4054


<ipython-input-8-906203b375ce>:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '24.954608935223273' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])
<ipython-input-8-906203b375ce>:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '272.8370576917745' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])
<ipython-input-8-906203b375ce>:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.7498107415961908' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])
<i

iForest


Model 1: KNeighborsClassifier
  Accuracy: 0.3735, Precision: 0.3754, Recall: 0.3735, F1-score: 0.3683

Model 2: KNeighborsClassifier
  Accuracy: 0.3639, Precision: 0.3669, Recall: 0.3639, F1-score: 0.3584

Model 3: KNeighborsClassifier
  Accuracy: 0.3564, Precision: 0.3574, Recall: 0.3564, F1-score: 0.3549

Model 4: DecisionTreeClassifier
  Accuracy: 0.3354, Precision: 0.2713, Recall: 0.3354, F1-score: 0.2618

Model 5: DecisionTreeClassifier
  Accuracy: 0.3314, Precision: 0.3312, Recall: 0.3314, F1-score: 0.3296

Model 6: DecisionTreeClassifier
  Accuracy: 0.4090, Precision: 0.4095, Recall: 0.4090, F1-score: 0.4086
SP


Model 1: KNeighborsClassifier
  Accuracy: 0.3738, Precision: 0.3758, Recall: 0.3738, F1-score: 0.3688

Model 2: KNeighborsClassifier
  Accuracy: 0.3680, Precision: 0.3705, Recall: 0.3680, F1-score: 0.3626

Model 3: KNeighborsClassifier
  Accuracy: 0.3638, Precision: 0.3646, Recall: 0.3638, F1-score: 0.3624

Model 4: DecisionTreeClassifier
  Accuracy: 0.3348, P

<ipython-input-8-906203b375ce>:65: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '23.02032924868804' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])
<ipython-input-8-906203b375ce>:65: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '251.68893311898924' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])
<ipython-input-8-906203b375ce>:65: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4.961842562641964' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])
<ip

iNNe


Model 1: KNeighborsClassifier
  Accuracy: 0.3730, Precision: 0.3746, Recall: 0.3730, F1-score: 0.3678

Model 2: KNeighborsClassifier
  Accuracy: 0.3659, Precision: 0.3686, Recall: 0.3659, F1-score: 0.3605

Model 3: KNeighborsClassifier
  Accuracy: 0.3585, Precision: 0.3592, Recall: 0.3585, F1-score: 0.3573

Model 4: DecisionTreeClassifier
  Accuracy: 0.3369, Precision: 0.2851, Recall: 0.3369, F1-score: 0.1760

Model 5: DecisionTreeClassifier
  Accuracy: 0.3332, Precision: 0.3066, Recall: 0.3332, F1-score: 0.2545

Model 6: DecisionTreeClassifier
  Accuracy: 0.4065, Precision: 0.4066, Recall: 0.4065, F1-score: 0.4059


In [15]:
best_algo_tame_outlier

'IQR'

In [16]:
x_copy_tame_outlier.Gender.nunique()

2

In [35]:
def SelectK(X, Y):
    selector = SelectKBest(f_classif, k="all")
    selector.fit(X.values, Y)
    selected_X = selector.transform(X.values)
    scores = selector.scores_
    return pd.DataFrame(selected_X, columns=X.columns[selector.get_support()]), Y, scores


def L1_Based(X, Y):
    lsvc = LinearSVC(C=0.01, penalty="l1", dual=False, max_iter=2000).fit(X.values, Y)
    model = SelectFromModel(lsvc, prefit=True)
    selected_X = model.transform(X.values)
    coefs = np.abs(lsvc.coef_).mean(axis=0)
    return pd.DataFrame(selected_X, columns=X.columns[model.get_support()]), Y, coefs


def TreeBased(X, Y):
    clf = ExtraTreesClassifier(n_estimators=50)
    clf.fit(X.values, Y)
    model = SelectFromModel(clf, prefit=True)
    selected_X = model.transform(X.values)
    importances = clf.feature_importances_
    return pd.DataFrame(selected_X, columns=X.columns[model.get_support()]), Y, importances


algorithm_functions_feature_selection = {
    "SelectK": SelectK,
    "L1_Based": L1_Based,
    "TreeBased": TreeBased,
}

In [36]:
def remove_constant_features(X):
    """Removes features that have the same value in all samples."""
    return X.loc[:, X.nunique() > 1]

def feature_selection(X,Y):
    acc_holder = {}
    x_copy = X.copy()
    y_copy = Y.copy()
    best_algo = None
    best_accuracy = 0
    final_scores=np.zeros(len(remove_constant_features(X).columns))
    for name, func in algorithm_functions_feature_selection.items():
        X_Adjusted,Y_Adjusted,Scores = func(remove_constant_features(x_copy.copy()), y_copy.copy())
        print(name)
        print()

        final_scores+=((Scores - Scores.min()) / (Scores.max() - Scores.min()))

        accuracy = train_and_evaluate(X_Adjusted, Y_Adjusted,expand=True)
        acc_holder[name] = accuracy

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_algo = name

    if best_algo:
        _,_,Scores = algorithm_functions_feature_selection[best_algo](remove_constant_features(x_copy.copy()), y_copy.copy())
        final_scores+=Scores/2

    percent = 0.5
    k = int(len(final_scores) * percent)
    top_column_indices = np.argsort(final_scores)[-k:][::-1]
    selected_columns = X.columns[top_column_indices]
    x_copy = x_copy[selected_columns]

    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

In [37]:
x_copy_feature_selection, y_copy_feature_selection, best_algo_feature_selection, best_accuracy_feature_selection,acc_holder_feature_selection=feature_selection(x_copy_tame_outlier,y_copy_tame_outlier)

SelectK


Model 1: KNeighborsClassifier
  Accuracy: 0.3738, Precision: 0.3758, Recall: 0.3738, F1-score: 0.3688

Model 2: KNeighborsClassifier
  Accuracy: 0.3680, Precision: 0.3705, Recall: 0.3680, F1-score: 0.3626

Model 3: KNeighborsClassifier
  Accuracy: 0.3638, Precision: 0.3646, Recall: 0.3638, F1-score: 0.3624

Model 4: DecisionTreeClassifier
  Accuracy: 0.3348, Precision: 0.3150, Recall: 0.3348, F1-score: 0.2606

Model 5: DecisionTreeClassifier
  Accuracy: 0.3301, Precision: 0.3238, Recall: 0.3301, F1-score: 0.2746

Model 6: DecisionTreeClassifier
  Accuracy: 0.4062, Precision: 0.4063, Recall: 0.4062, F1-score: 0.4061
L1_Based


Model 1: KNeighborsClassifier
  Accuracy: 0.3785, Precision: 0.3799, Recall: 0.3785, F1-score: 0.3729

Model 2: KNeighborsClassifier
  Accuracy: 0.3637, Precision: 0.3660, Recall: 0.3637, F1-score: 0.3576

Model 3: KNeighborsClassifier
  Accuracy: 0.3604, Precision: 0.3610, Recall: 0.3604, F1-score: 0.3585

Model 4: DecisionTreeClassifier
  Accuracy: 0.3

/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [43]:
x_copy_feature_selection.columns

Index(['Discharge Date_hour', 'Age', 'Hospital', 'Discharge Date_month',
       'Insurance Provider', 'Billing Amount', 'Name', 'Room Number', 'Doctor',
       'Discharge Date_day_of_week', 'Date of Admission_day',
       'Date of Admission_day_of_week'],
      dtype='object')